Reformat the datasets for the Flourish stories, e.g. number rounding, nice titles

In [1]:
import pandas as pd

In [18]:
avs_2024 = pd.read_csv("2024_averages.csv")

# Change proportion of skills to percentage
avs_2024.loc[avs_2024['variable']=='average_prop_green_skills','value'] = avs_2024.loc[avs_2024['variable']=='average_prop_green_skills','value']*100

# rename variables
avs_2024['variable'] = avs_2024['variable'].map({
    'average_prop_green_skills': "Average percentage of green skills",
    'average_ghg' :"Average GHG emissions",
       'average_green_timeshare': "Average percentage of time spent on green tasks",
    'number_job_ids': "Number of job adverts"})

avs_2024.to_csv("flourish_story/2024_averages.csv", index=False)

In [21]:
avs_quarterly = pd.read_csv("joined_quarter_averages.csv")

# Change proportion of skills to percentage
for country in ["All", "Wales", "England", "Scotland"]:
    avs_quarterly.loc[avs_quarterly['variable']=='average_prop_green_skills',
    country] = avs_quarterly.loc[avs_quarterly['variable']=='average_prop_green_skills', country]*100

# rename variables
avs_quarterly['variable'] = avs_quarterly['variable'].map({
    'average_prop_green_skills': "Average percentage of green skills",
    'average_ghg' :"Average GHG emissions",
       'average_green_timeshare': "Average percentage of time spent on green tasks",
    'number_job_ids': "Number of job adverts"})

avs_quarterly.to_csv("flourish_story/joined_quarter_averages.csv", index=False)

In [25]:
avs_itl3 = pd.read_csv("itl_3_name_averages.csv")

# Change proportion of skills to percentage
avs_itl3.loc[avs_itl3['variable']=='average_prop_green_skills','value'] = avs_itl3.loc[avs_itl3['variable']=='average_prop_green_skills','value']*100

# rename variables
avs_itl3['variable'] = avs_itl3['variable'].map({
    'average_prop_green_skills': "Average percentage of green skills",
    'average_ghg' :"Average GHG emissions",
       'average_green_timeshare': "Average percentage of time spent on green tasks",
    'number_job_ids': "Number of job adverts"})

avs_itl3.to_csv("flourish_story/itl_3_name_averages.csv", index=False)

## Find some examples of job titles for the sectors

In [31]:
import os
import polars as pl

In [33]:

titles_file = os.path.join(
    "s3://prinz-green-jobs/outputs/data/ojo_application",
    "deduplicated_sample/20241114/latest_update_20241114_titles.parquet",
)
titles_data = pl.read_parquet(titles_file)

In [34]:
green_sectors = ['Environmental', 'Welding/Plating/Pipefitting', 'Conservation/Environment', 'Other Energy',
                'Renewable Energy', 'Energy Advisor', 'Waste &amp; Recycling', 'Geotechnical', 'Waste Management',
                'Environmental Science', 'Water &amp; Environmental Consultancy', 'Ecology']

In [37]:
not_green_sectors = [
   'Other Transport &amp; Logistics', 
    'Vehicle Sales',
     'Transport Planner',
    'Airline',
     'Shipping',
         'Export Clerk',
     'Depot Manager',
        'Accounts Assistant',
        'Payroll',
     'Estate Agent',
     'Registered Mental Health Nurse',
     'Key Stage 2',
]

filter_list = green_sectors + not_green_sectors

In [38]:
filtered_titles_data = titles_data.filter(pl.col('sector').is_in(filter_list)).to_pandas()
len(filtered_titles_data)

261917

In [60]:
g.columns

Index(['id', 'company_raw', 'job_title_raw', 'job_location_raw', 'created',
       'type', 'sector', 'parent_sector', 'knowledge_domain', 'occupation'],
      dtype='object')

In [66]:
top_occs = {}
for i,g in filtered_titles_data.groupby('sector'):
    occs = g[g['occupation']!='']['occupation'].value_counts()[0:3].index.tolist()
    raw = g[g['occupation']!='']['job_title_raw'].value_counts()[0:3].index.tolist()
    top_occs[i] = occs
    print(f"- {i}: {occs}")
    # print(f"- {i}: {raw}")

- Accounts Assistant: ['Accounts Assistant', 'Finance Assistant', 'Finance Officer']
- Airline: ['Warehouse Operative', 'Hgv Driver', 'Delivery Driver']
- Conservation/Environment: ['Landscape Architect', 'Assistant Ecologist', 'Health Officer']
- Depot Manager: ['Transport Manager', 'Operations Manager', 'Depot Manager']
- Ecology: ['Principal Ecologist', 'Assistant Ecologist', 'Associate Director']
- Energy Advisor: ['Sustainability Consultant', 'Energy Consultant', 'Customer Advisor']
- Environmental: ['Acoustic Consultant', 'Quality Consultant', 'Town Planner']
- Environmental Science: ['Environmental Consultant', 'Health Officer', 'Environmental Advisor']
- Estate Agent: ['Estate Agent', 'Sales Negotiator', 'Apprentice Agent']
- Export Clerk: ['Export Clerk', 'Export Coordinator', 'Export Operator']
- Geotechnical: ['Geotechnical Engineer', 'Principal Engineer', 'Design Engineer']
- Key Stage 2: ['Primary Teacher', 'Teacher Assistant', 'Stage Teacher']
- Other Energy: ['Engineerin

## Get the salaries across all sectors and mean green measures

In [76]:
combined_green_measures_filename = os.path.join(
    "s3://prinz-green-jobs/outputs/data/ojo_application",
    "extracted_green_measures/analysis/20241121/combined_green_measures_and_meta.parquet",
)
combined_all_data_orig = pl.read_parquet(combined_green_measures_filename)

In [ ]:
combined_all_data_orig = combined_all_data_orig.with_columns((pl.col("itl_1_name")=="Wales").alias("in_wales"))

combined_all_data_orig = combined_all_data_orig.with_columns(  
    pl.col("itl_1_name").map_elements(lambda x: x if x in ["Wales", "Scotland", None] else "England").alias("country"),
)

In [88]:
df = combined_all_data_orig.join(titles_data[['id','parent_sector', 'sector']], left_on='job_id', right_on='id', how='left')

In [91]:
df_pd = df[['job_id', 'country', 'sector',
               'PROP_GREEN', 'GREEN TIMESHARE', 'INDUSTRY GHG PER UNIT EMISSIONS',
              'min_annualised_salary', 'max_annualised_salary']
].to_pandas()
df_pd['mid_annualised_salary'] = df_pd[['min_annualised_salary', 'max_annualised_salary']].mean(axis=1)

In [108]:
df_means = df_pd.groupby(['country', 'sector']).agg({
    'job_id': 'count',
    'PROP_GREEN': 'mean',
    'GREEN TIMESHARE': 'mean',
    'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
    'min_annualised_salary': 'mean',
    'max_annualised_salary': 'mean',  
    'mid_annualised_salary': 'mean',  
}).reset_index()

In [110]:
df_means.loc[df_means['sector'].isin(green_sectors), 'green_not_green_sector'] = 'Green sector'
df_means.loc[df_means['sector'].isin(not_green_sectors), 'green_not_green_sector'] = 'Not green sector'

In [111]:
df_means.rename(columns = {
    'PROP_GREEN': "Average percentage of green skills",
    'INDUSTRY GHG PER UNIT EMISSIONS' :"Average GHG emissions",
    'GREEN TIMESHARE': "Average percentage of time spent on green tasks",
    'job_id': "Number of job adverts",
    'min_annualised_salary': 'Average minimum salary',
    'max_annualised_salary': 'Average maximum salary',
    'mid_annualised_salary': 'Average salary mid point',
}, inplace=True)

In [112]:
df_means['Average percentage of green skills'] = df_means['Average percentage of green skills']*100
df_means[df_means["Number of job adverts"]>20].to_csv("flourish_story/sector_averages.csv", index=False)